### Imports

In [23]:
import xarray as xr
import gcsfs
import json
from dask.diagnostics import ProgressBar

from google.cloud import storage
from google.oauth2.credentials import Credentials

import sys
sys.path.append("../src/")
sys.path.append("../ocean_emulators_main/")
from ocean_emulators.dataset_validation import ds_input_validate, ds_raw_prediction_validate, ds_prediction_validate
from ocean_emulators.postprocessing import post_processor

## Data to leap

In [ ]:
with open(
    "/home/sd5313/.config/gcloud/application_default_credentials.json"
) as f:  # 🚨 make sure to enter the `.json` file from step 7
    token = json.load(f)
fs = gcsfs.GCSFileSystem(token=token)

#### Define Paths

In [34]:
local_truth_path = "/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr"
local_pred_path = "/pscratch/sd/s/suryad/Ocean_Emulator/Preds/2024-08-08_ConvNextUNetTrain3DEval3Dupscale2Epochs35Epoch30_Train_global_3D_Test_global_3D_all_N_train_4000_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_4000_rand_seed_1.zarr"

In [35]:
ds_truth = xr.open_zarr(local_truth_path)
ds_truth = ds_truth.isel(time=slice(4143, 4743)).isel(lev=slice(None, 19))
ds = xr.open_zarr(local_pred_path)
ds

<xarray.Dataset>
Dimensions:                        (time: 600, x: 180, y: 360, var: 77)
Dimensions without coordinates: time, x, y, var
Data variables:
    __xarray_dataarray_variable__  (time, x, y, var) float64 dask.array<chunksize=(75, 23, 45, 10), meta=np.ndarray>

#### Validation and Postprocessing


In [36]:
# ds_raw_prediction_validate(ds)
ds = post_processor(ds, ds_truth)
ds_prediction_validate(ds)

/pscratch/sd/s/suryad/Ocean_Emulator/notebooks/../ocean_emulators_main/ocean_emulators/postprocessing.py:20: UserWarning: Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream
  warnings.warn(
/pscratch/sd/s/suryad/Ocean_Emulator/notebooks/../ocean_emulators_main/ocean_emulators/dataset_validation.py:60: UserWarning: This checks nothing yet
  warnings.warn("This checks nothing yet")


In [ ]:
# mapper = fs.get_mapper(
#     "gs://leap-persistent/sd5313/convnext100MAddedSSTB_epoch-60_300_train-OM4v0.0_eval-OM4v0.0"
# )

# with ProgressBar():
#     ds.to_zarr(mapper)

## Data From Leap

In [5]:
# import an access token
# - option 1: read an access token from a file
with open("token.txt") as f:
    access_token = f.read().strip()

# setup a storage client using credentials
credentials = Credentials(access_token)
fs = gcsfs.GCSFileSystem(token=credentials)

#### Define paths

In [10]:
leap_path = 'gs://leap-persistent/jbusecke/ocean-emulators/OM4_5daily_v0.2.1.zarr'
local_path = '/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr'

In [ ]:
ds = xr.open_dataset(fs.get_mapper('leap_path'), engine='zarr', chunks={})
print(ds)

In [ ]:
# with ProgressBar():
#     ds.to_zarr(local_path, encoding = {v:{'compressor':None} for v in ds.variables},
#                 consolidated=True, mode='w')


#### Validation

In [11]:
data = xr.open_zarr(local_path)
ds_input_validate(data)